# Accident Hotspot Detection Using DBSCAN
## Google Colab Project Notebook

This notebook implements:
- CO1: Data loading, cleaning and feature preparation
- CO2: Logistic Regression and bias-variance analysis
- CO3: Decision Tree, Random Forest and feature importance
- CO4: DBSCAN clustering, Haversine distance and hotspot/anomaly detection
- CO5: Train/test split, cross-validation, GridSearchCV and evaluation
- CO6: Model packaging, prediction and basic drift detection

**Dataset:** `accident.csv`  
**Geographic scope:** Texas (`STATE = 48`)


In [ ]:
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

import os
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, roc_curve, confusion_matrix, classification_report

warnings.filterwarnings("ignore")
print("Libraries imported successfully.")

In [ ]:
# Upload accident.csv
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print("Uploaded file:", filename)

df = pd.read_csv(filename)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

In [ ]:
# CO1 — Dataset exploration
print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes)

print("\nMissing Values:")
display(df.isnull().sum().sort_values(ascending=False).head(20))

print("\nStatistical Summary:")
display(df.describe())

In [ ]:
# CO1 — Data cleaning
if "LONGITUDE" not in df.columns and "LONGITUD" in df.columns:
    df = df.rename(columns={"LONGITUD": "LONGITUDE"})

df = df[
    (df["LATITUDE"] < 90) &
    (df["LATITUDE"] > -90) &
    (df["LONGITUDE"].abs() < 180)
].copy()

print("Records after GPS cleaning:", len(df))
print("\nState counts:")
display(df["STATE"].value_counts().head(10))

In [ ]:
# CO1 — Select Texas
tx = df[df["STATE"] == 48].copy().reset_index(drop=True)
print("Texas accident records:", len(tx))
display(tx.head())

In [ ]:
# Save cleaned Texas dataset
os.makedirs("outputs", exist_ok=True)
tx.to_csv("outputs/texas_cleaned.csv", index=False)
print("Saved outputs/texas_cleaned.csv")

In [ ]:
# CO4 — Prepare coordinates for Haversine DBSCAN
coords = tx[["LATITUDE", "LONGITUDE"]].to_numpy()
coords_rad = np.radians(coords)

EARTH_RADIUS_KM = 6371.0088
print("Coordinate shape:", coords.shape)
print("Earth radius:", EARTH_RADIUS_KM, "km")

In [ ]:
# CO4 — K-distance elbow plot
MIN_SAMPLES = 5

neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES, metric="haversine")
neighbors.fit(coords_rad)

distances, indices = neighbors.kneighbors(coords_rad)
k_distances = np.sort(distances[:, -1])
k_dist_km = k_distances * EARTH_RADIUS_KM

plt.figure(figsize=(10, 6))
plt.plot(k_dist_km)
plt.xlabel("Points sorted by distance")
plt.ylabel("5th nearest neighbor distance (km)")
plt.title("K-Distance Elbow Plot for DBSCAN")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# CO4 — DBSCAN clustering
EPS_KM = 5.0
eps_rad = EPS_KM / EARTH_RADIUS_KM

dbscan = DBSCAN(
    eps=eps_rad,
    min_samples=MIN_SAMPLES,
    metric="haversine"
)

tx["cluster"] = dbscan.fit_predict(coords_rad)

n_clusters = len(set(tx["cluster"])) - (1 if -1 in tx["cluster"].values else 0)
n_noise = (tx["cluster"] == -1).sum()

print("DBSCAN Results")
print("=" * 40)
print("EPS:", EPS_KM, "km")
print("Minimum samples:", MIN_SAMPLES)
print("Clusters:", n_clusters)
print("Noise points:", n_noise)
print("Total records:", len(tx))

In [ ]:
# CO4 — Cluster distribution
cluster_counts = tx["cluster"].value_counts().sort_index()
display(cluster_counts.to_frame("accident_count"))

In [ ]:
# CO4 — Hotspot statistics
clustered = tx[tx["cluster"] != -1].copy()

summary = clustered.groupby("cluster").agg(
    accident_count=("cluster", "size"),
    avg_fatals=("FATALS", "mean"),
    total_fatals=("FATALS", "sum"),
    center_lat=("LATITUDE", "mean"),
    center_lon=("LONGITUDE", "mean")
).reset_index()

print("Number of hotspot clusters:", len(summary))
display(summary.head(10))

In [ ]:
# CO4 — Anomalous hotspot detection
for column in ["accident_count", "total_fatals"]:
    mean_value = summary[column].mean()
    std_value = summary[column].std(ddof=0)
    if std_value == 0:
        std_value = 1
    summary[column + "_z"] = (summary[column] - mean_value) / std_value

summary["is_anomalous_hotspot"] = (
    (summary["accident_count_z"] > 2) |
    (summary["total_fatals_z"] > 2)
)

summary = summary.sort_values("accident_count", ascending=False)

print("Anomalous hotspots:", summary["is_anomalous_hotspot"].sum())
display(summary.head(10))

In [ ]:
# Save hotspot summary
summary.to_csv("outputs/cluster_summary.csv", index=False)
print("Saved outputs/cluster_summary.csv")

In [ ]:
# CO4 — Hotspot map
plt.figure(figsize=(10, 8))

noise = tx[tx["cluster"] == -1]
plt.scatter(noise["LONGITUDE"], noise["LATITUDE"], s=10, alpha=0.4, label="Noise")

clustered = tx[tx["cluster"] != -1]
plt.scatter(
    clustered["LONGITUDE"],
    clustered["LATITUDE"],
    s=15,
    c=clustered["cluster"],
    cmap="tab20",
    alpha=0.8,
    label="Clusters"
)

anomalous_clusters = summary[summary["is_anomalous_hotspot"]]["cluster"]
anomalous_points = clustered[clustered["cluster"].isin(anomalous_clusters)]

plt.scatter(
    anomalous_points["LONGITUDE"],
    anomalous_points["LATITUDE"],
    s=60,
    facecolors="none",
    edgecolors="red",
    linewidths=1.5,
    label="Anomalous hotspot"
)

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Texas Accident Hotspots using DBSCAN")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# CO4 — Cluster size graph
plot_data = summary.copy()

plt.figure(figsize=(12, 6))
plt.bar(plot_data["cluster"].astype(str), plot_data["accident_count"])
plt.xlabel("Cluster ID")
plt.ylabel("Number of Accidents")
plt.title("Accident Count per Hotspot Cluster")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# CO5 — Create supervised-learning target
tx["MULTI_FATAL"] = (tx["FATALS"] >= 2).astype(int)

print("Target distribution:")
print(tx["MULTI_FATAL"].value_counts())

print("\nPercentage:")
print(tx["MULTI_FATAL"].value_counts(normalize=True).mul(100).round(2))

In [ ]:
# CO1 — Feature engineering
tx["hotspot_member"] = (tx["cluster"] != -1).astype(int)

FEATURE_COLS = [
    "WEATHER",
    "LGT_COND",
    "RUR_URB",
    "HOUR",
    "DAY_WEEK",
    "hotspot_member"
]
TARGET_COL = "MULTI_FATAL"

ml_data = tx[FEATURE_COLS + [TARGET_COL]].copy()

ml_data = ml_data[
    (ml_data["WEATHER"] < 90) &
    (ml_data["LGT_COND"] < 9)
].copy()

print("ML dataset shape:", ml_data.shape)
display(ml_data.head())

In [ ]:
# CO5 — Train / validation / test split
X = ml_data[FEATURE_COLS]
y = ml_data[TARGET_COL]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

In [ ]:
# CO2 — Feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")

In [ ]:
# CO2 — Logistic Regression bias-variance analysis
C_VALUES = [0.01, 0.1, 1, 10, 100]
bias_variance_results = []

for C in C_VALUES:
    model = LogisticRegression(
        C=C,
        penalty="l2",
        class_weight="balanced",
        max_iter=1000
    )
    model.fit(X_train_scaled, y_train)

    train_predictions = model.predict(X_train_scaled)
    val_predictions = model.predict(X_val_scaled)

    bias_variance_results.append({
        "C": C,
        "train_accuracy": accuracy_score(y_train, train_predictions),
        "validation_accuracy": accuracy_score(y_val, val_predictions)
    })

bv_df = pd.DataFrame(bias_variance_results)
display(bv_df)

In [ ]:
# CO2 — Bias-variance graph
plt.figure(figsize=(8, 5))
plt.plot(bv_df["C"], bv_df["train_accuracy"], marker="o", label="Training Accuracy")
plt.plot(bv_df["C"], bv_df["validation_accuracy"], marker="o", label="Validation Accuracy")
plt.xscale("log")
plt.xlabel("Regularization C")
plt.ylabel("Accuracy")
plt.title("Logistic Regression Bias-Variance Analysis")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Final Logistic Regression
best_C = bv_df.loc[bv_df["validation_accuracy"].idxmax(), "C"]

logreg = LogisticRegression(
    C=best_C,
    penalty="l2",
    class_weight="balanced",
    max_iter=1000
)
logreg.fit(X_train_scaled, y_train)

print("Best C:", best_C)
print("Logistic Regression trained.")

In [ ]:
# CO3 — Decision Tree
decision_tree = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

decision_tree.fit(X_train, y_train)
print("Decision Tree trained successfully.")

In [ ]:
# CO3 — Random Forest
random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight="balanced",
    random_state=42
)

random_forest.fit(X_train, y_train)
print("Random Forest trained successfully.")

In [ ]:
# CO3 — Feature importance
feature_importance = pd.Series(
    random_forest.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=False)

display(feature_importance.to_frame("importance"))

plt.figure(figsize=(8, 5))
feature_importance.sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# CO5 — 5-fold cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    random_forest,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("5-Fold ROC-AUC Scores:")
print(cv_scores)
print("\nMean ROC-AUC:", round(cv_scores.mean(), 4))
print("Standard Deviation:", round(cv_scores.std(), 4))

In [ ]:
# CO5 — Hyperparameter tuning
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 8, 12]
}

grid = GridSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)
print("\nBest Cross-validation ROC-AUC:", round(grid.best_score_, 4))

best_rf = grid.best_estimator_

In [ ]:
# CO5 — Model evaluation
def evaluate_model(name, model, X_test_data, y_test_data):
    predictions = model.predict(X_test_data)
    probabilities = model.predict_proba(X_test_data)[:, 1]

    return {
        "model": name,
        "accuracy": accuracy_score(y_test_data, predictions),
        "precision": precision_score(y_test_data, predictions, zero_division=0),
        "recall": recall_score(y_test_data, predictions, zero_division=0),
        "f1": f1_score(y_test_data, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test_data, probabilities),
        "pr_auc": average_precision_score(y_test_data, probabilities)
    }

results = []

results.append(evaluate_model("Logistic Regression", logreg, X_test_scaled, y_test))
results.append(evaluate_model("Decision Tree", decision_tree, X_test, y_test))
results.append(evaluate_model("Random Forest", best_rf, X_test, y_test))

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
# Save model comparison
results_df.to_csv("outputs/co5_model_comparison.csv", index=False)
print("Saved outputs/co5_model_comparison.csv")

In [ ]:
# CO5 — ROC curves
plt.figure(figsize=(8, 6))

models = [
    ("Logistic Regression", logreg, X_test_scaled),
    ("Decision Tree", decision_tree, X_test),
    ("Random Forest", best_rf, X_test)
]

for name, model, X_data in models:
    probabilities = model.predict_proba(X_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f"{name} AUC={auc:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Multi-Fatality Prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# CO5 — Confusion matrix
predictions = best_rf.predict(X_test)
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Single Fatal", "Multi Fatal"],
    yticklabels=["Single Fatal", "Multi Fatal"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Random Forest")
plt.show()

print(classification_report(
    y_test,
    predictions,
    target_names=["Single Fatal", "Multi Fatal"],
    zero_division=0
))

In [ ]:
# CO6 — Model packaging
model_package = {
    "model": best_rf,
    "scaler": scaler,
    "features": FEATURE_COLS
}

joblib.dump(model_package, "outputs/model_package.joblib")
print("Model saved: outputs/model_package.joblib")

In [ ]:
# CO6 — Sample prediction
loaded_package = joblib.load("outputs/model_package.joblib")
loaded_model = loaded_package["model"]
features = loaded_package["features"]

sample = pd.DataFrame([{
    "WEATHER": 1,
    "LGT_COND": 2,
    "RUR_URB": 2,
    "HOUR": 22,
    "DAY_WEEK": 6,
    "hotspot_member": 1
}])

sample = sample[features]

probability = loaded_model.predict_proba(sample)[0, 1]
prediction = loaded_model.predict(sample)[0]

display(sample)
print("Multi-fatality probability:", round(probability, 4))
print("Prediction:", "MULTI-FATALITY RISK" if prediction == 1 else "SINGLE-FATALITY RISK")

In [ ]:
# CO6 — Basic data drift detection
def check_feature_drift(training_data, new_data, threshold=2.0):
    report = {}

    for column in training_data.columns:
        train_mean = training_data[column].mean()
        train_std = training_data[column].std()
        new_mean = new_data[column].mean()

        if train_std == 0:
            continue

        z_shift = (new_mean - train_mean) / train_std

        report[column] = {
            "train_mean": round(train_mean, 3),
            "new_mean": round(new_mean, 3),
            "z_shift": round(z_shift, 3),
            "drift_flag": abs(z_shift) > threshold
        }

    return report

drift_report = check_feature_drift(X_train, X_test)
display(pd.DataFrame(drift_report).T)

In [ ]:
# Save drift report
with open("outputs/co6_drift_report.json", "w") as file:
    json.dump(drift_report, file, indent=4)

print("Saved outputs/co6_drift_report.json")

In [ ]:
# Final output list
print("PROJECT OUTPUT FILES")
print("=" * 50)

for root, dirs, files_list in os.walk("outputs"):
    for file in files_list:
        print(os.path.join(root, file))

In [ ]:
# Download all generated outputs
!zip -r accident_hotspot_outputs.zip outputs > /dev/null
files.download("accident_hotspot_outputs.zip")